# Ads Revenue EDA (SQLite)

This notebook reads a synthetic Meta-style ads revenue dataset from a local SQLite database (`meta_ads.db`) and performs exploratory data analysis (EDA):
- data loading + joins
- basic data quality checks
- core KPIs (spend, revenue, ROAS, CTR, CVR)
- breakdowns by product / advertiser / industry
- time trends + a few plots


In [ ]:
import sqlite3
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

DB_PATH = Path("meta_ads.db")
assert DB_PATH.exists(), f"DB not found at {DB_PATH.resolve()}"

conn = sqlite3.connect(DB_PATH)
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 140)


In [ ]:
# Load tables
tx = pd.read_sql_query("SELECT * FROM ad_transactions", conn)
advertisers = pd.read_sql_query("SELECT * FROM advertisers", conn)
products = pd.read_sql_query("SELECT * FROM ad_products", conn)

tx["event_ts"] = pd.to_datetime(tx["event_ts"], errors="coerce")

df = (
    tx.merge(advertisers, on="advertiser_id", how="left")
      .merge(products, on="product_id", how="left")
)

tx.shape, advertisers.shape, products.shape, df.shape


In [ ]:
# Data quality quick checks
summary = pd.DataFrame(
    {
        "rows": [len(df)],
        "min_event_ts": [df["event_ts"].min()],
        "max_event_ts": [df["event_ts"].max()],
        "null_event_ts": [df["event_ts"].isna().sum()],
        "null_advertiser": [df["advertiser_name"].isna().sum()],
        "null_product": [df["product_name"].isna().sum()],
    }
)

violations = {
    "clicks_gt_impressions": int((df["clicks"] > df["impressions"]).sum()),
    "conversions_gt_clicks": int((df["conversions"] > df["clicks"]).sum()),
    "negative_spend": int((df["spend_usd"] < 0).sum()),
    "negative_revenue": int((df["revenue_usd"] < 0).sum()),
}

summary, violations


In [ ]:
# Core KPIs
total_spend = df["spend_usd"].sum()
total_revenue = df["revenue_usd"].sum()
overall_roas = total_revenue / total_spend

# Weighted rates
total_impressions = df["impressions"].sum()
total_clicks = df["clicks"].sum()
total_conversions = df["conversions"].sum()

ctr = total_clicks / total_impressions
cvr = total_conversions / max(total_clicks, 1)
cpm = (total_spend / total_impressions) * 1000
cpc = total_spend / max(total_clicks, 1)
cpa = total_spend / max(total_conversions, 1)

pd.DataFrame(
    {
        "total_spend_usd": [total_spend],
        "total_revenue_usd": [total_revenue],
        "overall_roas": [overall_roas],
        "ctr": [ctr],
        "cvr": [cvr],
        "cpm": [cpm],
        "cpc": [cpc],
        "cpa": [cpa],
    }
).round(4)


In [ ]:
# Breakdown: revenue/spend by product
by_product = (
    df.groupby(["product_name", "placement", "objective", "pricing_model"], as_index=False)
      .agg(spend_usd=("spend_usd", "sum"), revenue_usd=("revenue_usd", "sum"), impressions=("impressions", "sum"), clicks=("clicks", "sum"), conversions=("conversions", "sum"))
)
by_product["roas"] = by_product["revenue_usd"] / by_product["spend_usd"]
by_product.sort_values("revenue_usd", ascending=False).head(10)


In [ ]:
# Breakdown: top advertisers
by_adv = (
    df.groupby(["advertiser_name", "industry", "account_tier", "hq_country"], as_index=False)
      .agg(spend_usd=("spend_usd", "sum"), revenue_usd=("revenue_usd", "sum"), impressions=("impressions", "sum"), clicks=("clicks", "sum"), conversions=("conversions", "sum"))
)
by_adv["roas"] = by_adv["revenue_usd"] / by_adv["spend_usd"]
by_adv.sort_values("revenue_usd", ascending=False).head(10)


In [ ]:
# Industry + tier comparisons
by_industry = (
    df.groupby(["industry", "account_tier"], as_index=False)
      .agg(spend_usd=("spend_usd", "sum"), revenue_usd=("revenue_usd", "sum"), impressions=("impressions", "sum"), clicks=("clicks", "sum"), conversions=("conversions", "sum"))
)
by_industry["roas"] = by_industry["revenue_usd"] / by_industry["spend_usd"]
by_industry.sort_values(["revenue_usd"], ascending=False).head(20)


In [ ]:
# Time series (daily) + moving average
daily = (
    df.assign(day=df["event_ts"].dt.floor("D"))
      .groupby("day", as_index=False)
      .agg(spend_usd=("spend_usd", "sum"), revenue_usd=("revenue_usd", "sum"))
      .sort_values("day")
)
daily["roas"] = daily["revenue_usd"] / daily["spend_usd"]
daily["revenue_ma7"] = daily["revenue_usd"].rolling(7).mean()
daily.tail(10)


In [ ]:
# Plot: revenue by product
plt.figure(figsize=(10, 5))
tmp = by_product.sort_values("revenue_usd", ascending=False).head(10)
plt.bar(tmp["product_name"], tmp["revenue_usd"])
plt.xticks(rotation=30, ha="right")
plt.title("Top products by revenue (USD)")
plt.ylabel("Revenue (USD)")
plt.tight_layout()
plt.show()


In [ ]:
# Plot: revenue trend
plt.figure(figsize=(10, 5))
plt.plot(daily["day"], daily["revenue_usd"], label="Daily revenue")
plt.plot(daily["day"], daily["revenue_ma7"], label="7-day MA")
plt.title("Revenue over time")
plt.ylabel("Revenue (USD)")
plt.xlabel("Day")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
conn.close()
